# **Praktikum 5: Pembobotan TF-IDF**

**Langkah praktikum: Penerapan TF-IDF di Python**

Praktikum hari ini akan mempelajari:
1. Hitung manual TF-IDF menggunakan Spreadsheet
2. Hitung TF-IDF menggunakan Python

**Silakan copy file Spreadsheet dan Colab ke akun Google masing-masing**

👇👇👇 Akses file Spreadsheet (jangan lupa disalin ke akun masing-masing dulu sebelum diedit!) 👇👇👇

https://docs.google.com/spreadsheets/d/1irh7oxqxlCxC96tEG03OLLd7crOG-Ih3/edit?usp=sharing&ouid=108501208574741240775&rtpof=true&sd=true

Dokumen = “Kebun binatang itu terlihat sangat luas. Berbagai macam binatang terdapat di dalamnya. Binatang-binatang di dalam kebun tersebut terlihat gemuk.”


## **Preparation**

**1. Import library yang akan digunakan**

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import pandas as pd


**2. Siapkan dataset**

In [2]:
!wget 'https://raw.githubusercontent.com/jiorjioo/PBA2025/refs/heads/main/cleaned_tweet_efisiensi.csv'

--2026-11-03 10:24:27--  https://raw.githubusercontent.com/jiorjioo/PBA2025/refs/heads/main/cleaned_tweet_efisiensi.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1142103 (1.1M) [text/plain]
Saving to: ‘cleaned_tweet_efisiensi.csv’

cleaned_tweet_efisi 100%[===================>]   1.09M  --.-KB/s    in 0.03s   

2026-11-03 10:24:27 (35.5 MB/s) - ‘cleaned_tweet_efisiensi.csv’ saved [1142103/1142103]



In [4]:
df = pd.read_csv("cleaned_tweet_efisiensi.csv")
df

,Unnamed: 0,created_at,favorite_count,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,full_text,cleaned_text,stemmed_text,tokenized_text
0,0,Thu Mar 13 23:31:50 +0000 2025,0,NaN,in,Papua Barat,0,1,0,Danantara Indonesia berkomitmen untuk mendoron...,danantara indonesia berkomitmen mendorong tran...,danantara indonesia komitmen dorong transforma...,"['danantara', 'indonesia', 'komitmen', 'dorong..."
1,1,Thu Mar 13 23:31:20 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Kalau bs efisiensi Logikanya a...,bs efisiensi logikanya anggaran surplus,bs efisiensi logika anggar surplus,"['bs', 'efisiensi', 'logika', 'anggar', 'surpl..."
2,2,Thu Mar 13 23:29:40 +0000 2025,0,CNNIndonesia,in,NaN,0,0,0,@CNNIndonesia Apa sih buk! Noh efisiensi tapi ...,sih buk noh efisiensi pejabat asik asik,sih buk noh efisiensi jabat asik asik,"['sih', 'buk', 'noh', 'efisiensi', 'jabat', 'a..."
3,3,Thu Mar 13 23:27:17 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,efisiensi lari danantaro makan bergizi gratis ...,efisiensi lari danantaro makan gizi gratis bik...,"['efisiensi', 'lari', 'danantaro', 'makan', 'g..."
4,4,Thu Mar 13 23:22:25 +0000 2025,0,geloraco,in,NaN,0,0,0,@geloraco Kiw kiwww hiburan baru buat dana bar...,kiw kiwww hiburan dana efisiensi diperpanjang,kiw kiwww hibur dana efisiensi panjang,"['kiw', 'kiwww', 'hibur', 'dana', 'efisiensi',..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2510,2510,Sun Mar 09 05:21:25 +0000 2025,1,islah_bahrawi,in,Indonesia,0,0,0,@islah_bahrawi Maju PSI!!! Hidup Efisiensi!! H...,maju psi hidup efisiensi hidup jokowi mbelgedes,maju psi hidup efisiensi hidup jokowi mbelgedes,"['maju', 'psi', 'hidup', 'efisiensi', 'hidup',..."
2511,2511,Sun Mar 09 05:20:53 +0000 2025,0,NaN,in,Shattering skies above,0,0,0,Indonesia jam 10 lampu harus udah mati semua g...,indonesia jam lampu udah mati gara efisiensi a...,indonesia jam lampu udah mati gara efisiensi a...,"['indonesia', 'jam', 'lampu', 'udah', 'mati', ..."
2512,2512,Sun Mar 09 05:19:41 +0000 2025,0,NaN,in,Jahannam Lt. Underground,0,1,0,Kebenaran Efektif adalah hal yang tercermin da...,kebenaran efektif tercermin tindakan sikap sik...,benar efektif cermin tindak sikap sikap angkuh...,"['benar', 'efektif', 'cermin', 'tindak', 'sika..."
2513,2513,Sun Mar 09 05:18:33 +0000 2025,0,EricRicaldo,in,NaN,0,0,0,@EricRicaldo @prabowo Presidennya Paham efisie...,presidennya paham efisiensi ga gengsi tau ah ono,presiden paham efisiensi ga gengsi tau ah ono,"['presiden', 'paham', 'efisiensi', 'ga', 'geng..."


Praktik kali ini ingin menerapkan TF-IDF sederhana dari real data yang sudah dipelajari di pertemuan sebelumnya. Untuk pemahaman dasar, data yang diambil adalah data dari kolom stemmed_text yang jumlah tokennya kurang dari 10.

**1. Ketahui dulu total kata pada seluruh baris di kolom stemmed_text**

In [5]:
df['totalwords'] = df['stemmed_text'].str.count(' ') + 1
df

,Unnamed: 0,created_at,favorite_count,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,full_text,cleaned_text,stemmed_text,tokenized_text,totalwords
0,0,Thu Mar 13 23:31:50 +0000 2025,0,NaN,in,Papua Barat,0,1,0,Danantara Indonesia berkomitmen untuk mendoron...,danantara indonesia berkomitmen mendorong tran...,danantara indonesia komitmen dorong transforma...,"['danantara', 'indonesia', 'komitmen', 'dorong...",26.0
1,1,Thu Mar 13 23:31:20 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Kalau bs efisiensi Logikanya a...,bs efisiensi logikanya anggaran surplus,bs efisiensi logika anggar surplus,"['bs', 'efisiensi', 'logika', 'anggar', 'surpl...",5.0
2,2,Thu Mar 13 23:29:40 +0000 2025,0,CNNIndonesia,in,NaN,0,0,0,@CNNIndonesia Apa sih buk! Noh efisiensi tapi ...,sih buk noh efisiensi pejabat asik asik,sih buk noh efisiensi jabat asik asik,"['sih', 'buk', 'noh', 'efisiensi', 'jabat', 'a...",7.0
3,3,Thu Mar 13 23:27:17 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,efisiensi lari danantaro makan bergizi gratis ...,efisiensi lari danantaro makan gizi gratis bik...,"['efisiensi', 'lari', 'danantaro', 'makan', 'g...",12.0
4,4,Thu Mar 13 23:22:25 +0000 2025,0,geloraco,in,NaN,0,0,0,@geloraco Kiw kiwww hiburan baru buat dana bar...,kiw kiwww hiburan dana efisiensi diperpanjang,kiw kiwww hibur dana efisiensi panjang,"['kiw', 'kiwww', 'hibur', 'dana', 'efisiensi',...",6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2510,2510,Sun Mar 09 05:21:25 +0000 2025,1,islah_bahrawi,in,Indonesia,0,0,0,@islah_bahrawi Maju PSI!!! Hidup Efisiensi!! H...,maju psi hidup efisiensi hidup jokowi mbelgedes,maju psi hidup efisiensi hidup jokowi mbelgedes,"['maju', 'psi', 'hidup', 'efisiensi', 'hidup',...",7.0
2511,2511,Sun Mar 09 05:20:53 +0000 2025,0,NaN,in,Shattering skies above,0,0,0,Indonesia jam 10 lampu harus udah mati semua g...,indonesia jam lampu udah mati gara efisiensi a...,indonesia jam lampu udah mati gara efisiensi a...,"['indonesia', 'jam', 'lampu', 'udah', 'mati', ...",10.0
2512,2512,Sun Mar 09 05:19:41 +0000 2025,0,NaN,in,Jahannam Lt. Underground,0,1,0,Kebenaran Efektif adalah hal yang tercermin da...,kebenaran efektif tercermin tindakan sikap sik...,benar efektif cermin tindak sikap sikap angkuh...,"['benar', 'efektif', 'cermin', 'tindak', 'sika...",23.0
2513,2513,Sun Mar 09 05:18:33 +0000 2025,0,EricRicaldo,in,NaN,0,0,0,@EricRicaldo @prabowo Presidennya Paham efisie...,presidennya paham efisiensi ga gengsi tau ah ono,presiden paham efisiensi ga gengsi tau ah ono,"['presiden', 'paham', 'efisiensi', 'ga', 'geng...",8.0


**2. Ambil (filter) data yang hanya memiliki jumlah kata <= 10**

In [6]:
df = df[df['totalwords'] <= 10].reset_index()
df

,index,Unnamed: 0,created_at,favorite_count,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,full_text,cleaned_text,stemmed_text,tokenized_text,totalwords
0,1,1,Thu Mar 13 23:31:20 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Kalau bs efisiensi Logikanya a...,bs efisiensi logikanya anggaran surplus,bs efisiensi logika anggar surplus,"['bs', 'efisiensi', 'logika', 'anggar', 'surpl...",5.0
1,2,2,Thu Mar 13 23:29:40 +0000 2025,0,CNNIndonesia,in,NaN,0,0,0,@CNNIndonesia Apa sih buk! Noh efisiensi tapi ...,sih buk noh efisiensi pejabat asik asik,sih buk noh efisiensi jabat asik asik,"['sih', 'buk', 'noh', 'efisiensi', 'jabat', 'a...",7.0
2,4,4,Thu Mar 13 23:22:25 +0000 2025,0,geloraco,in,NaN,0,0,0,@geloraco Kiw kiwww hiburan baru buat dana bar...,kiw kiwww hiburan dana efisiensi diperpanjang,kiw kiwww hibur dana efisiensi panjang,"['kiw', 'kiwww', 'hibur', 'dana', 'efisiensi',...",6.0
3,5,5,Thu Mar 13 23:21:18 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Efisiensi buat stafsus,efisiensi stafsus,efisiensi stafsus,"['efisiensi', 'stafsus']",2.0
4,12,12,Thu Mar 13 23:12:10 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Tuh kaaannn efisiensi semakin ...,tuh kaaannn efisiensi nih doi mo bikin penjara...,tuh kaaannn efisiensi nih doi mo bikin penjara...,"['tuh', 'kaaannn', 'efisiensi', 'nih', 'doi', ...",10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1437,2507,2507,Sun Mar 09 05:29:17 +0000 2025,0,neVerAl0nely,in,"Bandung, Jawa Barat",0,0,0,@neVerAl0nely ini gada penjaga rel karena penj...,gada penjaga rel penjaga rel diberhentikan ken...,gada jaga rel jaga rel henti kena efisiensi,"['gada', 'jaga', 'rel', 'jaga', 'rel', 'henti'...",8.0
1438,2508,2508,Sun Mar 09 05:25:37 +0000 2025,0,AuslanderDon,in,"Yogyakarta, Indonesia",0,1,0,@AuslanderDon Pura-pura salah tafsir hasil RDP...,purapura salah tafsir hasil rdp yg ditafsirkan...,purapura salah tafsir hasil rdp yg tafsir rdp ...,"['purapura', 'salah', 'tafsir', 'hasil', 'rdp'...",9.0
1439,2510,2510,Sun Mar 09 05:21:25 +0000 2025,1,islah_bahrawi,in,Indonesia,0,0,0,@islah_bahrawi Maju PSI!!! Hidup Efisiensi!! H...,maju psi hidup efisiensi hidup jokowi mbelgedes,maju psi hidup efisiensi hidup jokowi mbelgedes,"['maju', 'psi', 'hidup', 'efisiensi', 'hidup',...",7.0
1440,2511,2511,Sun Mar 09 05:20:53 +0000 2025,0,NaN,in,Shattering skies above,0,0,0,Indonesia jam 10 lampu harus udah mati semua g...,indonesia jam lampu udah mati gara efisiensi a...,indonesia jam lampu udah mati gara efisiensi a...,"['indonesia', 'jam', 'lampu', 'udah', 'mati', ...",10.0


In [7]:
df.sort_values(by='totalwords', ascending=True)

,index,Unnamed: 0,created_at,favorite_count,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,full_text,cleaned_text,stemmed_text,tokenized_text,totalwords
1027,1711,1711,Mon Mar 10 22:52:42 +0000 2025,0,tanganjawh,es,soc,0,1,0,@tanganjawh @idextratime @sport Efisiensi,efisiensi,efisiensi,['efisiensi'],1.0
51,94,94,Thu Mar 13 18:36:38 +0000 2025,0,yeojunoona,in,NaN,0,1,0,@adorableehann lama lama kasihan,kasihan,kasihan,['kasihan'],1.0
54,100,100,Thu Mar 13 18:19:23 +0000 2025,0,Sakuvya,in,𝓨𝓾𝓱𝓪𝓽𝓪𝓵𝓮 𐙚 (2007),0,1,0,@Sakuvya Lagi efisiensi katanya ️️,efisiensi,efisiensi,['efisiensi'],1.0
590,977,977,Wed Mar 12 02:45:30 +0000 2025,0,NaN,es,gapyear²⁴,0,0,0,efisiensi,efisiensi,efisiensi,['efisiensi'],1.0
527,866,866,Wed Mar 12 07:08:20 +0000 2025,0,nxtalgictee,es,she/her²⁰,0,0,0,@nxtalgictee efisiensi,efisiensi,efisiensi,['efisiensi'],1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1063,1779,1779,Mon Mar 10 21:08:20 +0000 2025,1,sutanmangara,in,.,0,0,0,@sutanmangara Efek efisiensi.. Semua narapidan...,efek efisiensi narapidana dilepas krn negara g...,efek efisiensi narapidana lepas krn negara gak...,"['efek', 'efisiensi', 'narapidana', 'lepas', '...",10.0
994,1645,1645,Tue Mar 11 02:20:03 +0000 2025,3,NaN,in,NaN,2,0,0,Konsolidasi Danantara: Meningkatkan Efisiensi ...,konsolidasi danantara meningkatkan efisiensi d...,konsolidasi danantara tingkat efisiensi daya s...,"['konsolidasi', 'danantara', 'tingkat', 'efisi...",10.0
993,1639,1639,Tue Mar 11 02:26:11 +0000 2025,0,NaN,in,NaN,0,0,0,Konsolidasi Danantara: Meningkatkan Efisiensi ...,konsolidasi danantara meningkatkan efisiensi d...,konsolidasi danantara tingkat efisiensi daya s...,"['konsolidasi', 'danantara', 'tingkat', 'efisi...",10.0
998,1658,1658,Tue Mar 11 02:01:27 +0000 2025,50,NOTASLIMBOY,in,NaN,0,0,0,@NOTASLIMBOY Satu negara disuruh efisiensi sen...,negara disuruh efisiensi bikin latsar magelang...,negara suruh efisiensi bikin latsar magelang m...,"['negara', 'suruh', 'efisiensi', 'bikin', 'lat...",10.0


**3. Ambil sebagian data (subset) untuk praktikum hari ini menggunakan iloc**

In [8]:
subset=df.iloc[[41, 55, 99, 118, 142],[12]]
# Dengan iloc, kita bisa mengambil data secara spesifik berdasarkan index dan kolom
# Data yang diambil adalah df index ke-41, 55, 99, 118, 142 dari kolom ke-11 yakni stemmed_text

subset

,cleaned_text
41,kena efisiensi
55,sabar ya azril wowo kena efisiensi bjirgwe jua...
99,asli weh inimah bener kena efisiensi
118,lol efisiensi nya biar enak menjabat tolol
142,wooahae beneran kena efisiensi wkwwk


**4. Konversi data subset di atas ke dalam bentuk list**

In [12]:
list_dokumen = [item[0] for item in subset.values.tolist()]
list_dokumen

['kena efisiensi',
 'sabar ya azril wowo kena efisiensi bjirgwe jual onglipengs',
 'asli weh inimah bener kena efisiensi',
 'lol efisiensi nya biar enak menjabat tolol',
 'wooahae beneran kena efisiensi wkwwk']

## **Penerapan TF-IDF**

### **A. Analisis dokumen/corpus yang akan digunakan**

**1. Menghitung jumlah unique term/vocab**

In [15]:
CV = CountVectorizer()
term_fit = CV.fit(list_dokumen)

print(len(term_fit.vocabulary_))


22


**2. Menampilkan unique term/vocab dan indexnya**

In [16]:
print(term_fit.vocabulary_)

{'kena': 10, 'efisiensi': 6, 'sabar': 15, 'ya': 21, 'azril': 1, 'wowo': 20, 'bjirgwe': 5, 'jual': 9, 'onglipengs': 14, 'asli': 0, 'weh': 17, 'inimah': 8, 'bener': 2, 'lol': 11, 'nya': 13, 'biar': 4, 'enak': 7, 'menjabat': 12, 'tolol': 16, 'wooahae': 19, 'beneran': 3, 'wkwwk': 18}


In [18]:
# Kode tambahan untuk melihat daftar term unik (unique term) secara urut dan lebih rapi
# Angka menunjukkan index/urutan kata. Kata telah disusun otomatis sesuai abjad

for word, index in sorted(term_fit.vocabulary_.items(), key=lambda item: item[1]):
    print(f"{index}: {word}")

0: asli
1: azril
2: bener
3: beneran
4: biar
5: bjirgwe
6: efisiensi
7: enak
8: inimah
9: jual
10: kena
11: lol
12: menjabat
13: nya
14: onglipengs
15: sabar
16: tolol
17: weh
18: wkwwk
19: wooahae
20: wowo
21: ya


In [19]:
# Menampilkan daftar unique term/vocab
print(term_fit.get_feature_names_out())


['asli' 'azril' 'bener' 'beneran' 'biar' 'bjirgwe' 'efisiensi' 'enak'
 'inimah' 'jual' 'kena' 'lol' 'menjabat' 'nya' 'onglipengs' 'sabar'
 'tolol' 'weh' 'wkwwk' 'wooahae' 'wowo' 'ya']


### **B. Proses TF-IDF**

**1. Menampilkan term frequency dari keseluruhan dokumen**

In [20]:
term_frequency_all = term_fit.transform(list_dokumen)
print(term_frequency_all)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 29 stored elements and shape (5, 22)>
  Coords	Values
  (0, 6)	1
  (0, 10)	1
  (1, 1)	1
  (1, 5)	1
  (1, 6)	1
  (1, 9)	1
  (1, 10)	1
  (1, 14)	1
  (1, 15)	1
  (1, 20)	1
  (1, 21)	1
  (2, 0)	1
  (2, 2)	1
  (2, 6)	1
  (2, 8)	1
  (2, 10)	1
  (2, 17)	1
  (3, 4)	1
  (3, 6)	1
  (3, 7)	1
  (3, 11)	1
  (3, 12)	1
  (3, 13)	1
  (3, 16)	1
  (4, 3)	1
  (4, 6)	1
  (4, 10)	1
  (4, 18)	1
  (4, 19)	1


Makna dari daftar di atas:

- **0, 1, 2, dst di depan** >> menunjukkan nomor kalimatnya (kan ada 5 kalimat tuh yg buat jd bahan dokumen)
- **angka kedua** >> menunjukkan index kata
- **Values** >> jumlah kemunculan kata tersebut dalam kalimat ke-n

misal pada (0,3) memiliki Values 2. maknanya kata index ke-3 muncul di kalimat index ke-0 sebanyak 2 kali.
- kata index ke-3 = efisiensi
- kalimat ke-0 = efisiensi emang budaya orang indonesia kek efisiensi

**2. Menampilkan term atau feature tertentu berdasarkan indeks termvocabulary**

In [21]:
print(term_fit.get_feature_names_out()[3])
print(term_fit.get_feature_names_out()[17])
print(term_fit.get_feature_names_out()[5])

beneran
weh
bjirgwe


##### **3.TF IDF untuk dokumen/kalimat tertentu**

**a. Menentukan dokumen uji coba yaitu dokumen dengan indeks ke-0, kemudian dimasukkan ke dalam variabel komentar_tf**

In [22]:
komentar_tf = list_dokumen[0]
print(komentar_tf)

kena efisiensi


**b. Menampilkan term frequency pada dokumen yang disimpan dalam variabel komentar_tf**

In [23]:
#TF untuk semua dokumen
term_frequency = term_fit.transform([komentar_tf])
print(term_frequency)


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2 stored elements and shape (1, 22)>
  Coords	Values
  (0, 6)	1
  (0, 10)	1


**c. Menghitung TF-IDF dari satu kalimat tertentu**

In [24]:
# Menghitung nilai TF-IDF untuk dokumen/kalimat tertentu
komentar_term = term_fit.transform(list_dokumen)
tfidf_transformer = TfidfTransformer().fit(komentar_term)
tfidf = tfidf_transformer.transform(term_frequency)

print(round(tfidf,3))



<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 22)>
  Coords	Values
  (0, 6)	0.646
  (0, 10)	0.764


**d. Mengetahui nilai IDF dari seluruh vocab/term**

In [25]:
print(tfidf_transformer.idf_)


[2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 1.         2.09861229 2.09861229 2.09861229 1.18232156 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229]


**e. Mengetahui nilai IDF untuk vocab/term tertentu**

In [26]:
print(tfidf_transformer.idf_[term_fit.vocabulary_['efisiensi']])
print(tfidf_transformer.idf_[term_fit.vocabulary_['enak']])

1.0
2.09861228866811


**f. Menghitung TF-IDF untuk seluruh dokumen (seluruh 5 komentar)**

In [27]:
# Menghitung nilai TF-IDF untuk seluruh dokumen
komentar_tfidf = tfidf_transformer.transform(komentar_term)
print(round(komentar_tfidf,3))



<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 29 stored elements and shape (5, 22)>
  Coords	Values
  (0, 6)	0.646
  (0, 10)	0.764
  (1, 1)	0.364
  (1, 5)	0.364
  (1, 6)	0.173
  (1, 9)	0.364
  (1, 10)	0.205
  (1, 14)	0.364
  (1, 15)	0.364
  (1, 20)	0.364
  (1, 21)	0.364
  (2, 0)	0.469
  (2, 2)	0.469
  (2, 6)	0.224
  (2, 8)	0.469
  (2, 10)	0.264
  (2, 17)	0.469
  (3, 4)	0.401
  (3, 6)	0.191
  (3, 7)	0.401
  (3, 11)	0.401
  (3, 12)	0.401
  (3, 13)	0.401
  (3, 16)	0.401
  (4, 3)	0.531
  (4, 6)	0.253
  (4, 10)	0.299
  (4, 18)	0.531
  (4, 19)	0.531


# **Postest**


1. Pilih 5 kalimat berbeda (subset) dari kumpulan data dengan token kurang dari 10. Dengan catatan, kalimat yang dipilih tidak boleh sama dengan langkah praktikum.
2. Duplikasikan sheet1 dari template manual. Lalu hitung TF-IDF dari dokumen post test mu.
3. Terapkan ke dalam kode langkah praktikum, mulai dari poin B. pada poin “mengetahui nilai IDF untuk vocab term tertentu” silahkan pilih 2 vocab bebas berdasarkan dokumenmu.
4. Apakah ada perbedaan hasil antara hasil TF-IDF yang dihitung secara manual menggunakan rumus dengan hasil TF-IDF coding? Kalau memang ada, analisis dan jelaskan perbedaannya.

Pilih 5 kalimat berbeda (subset) dari kumpulan data dengan token kurang dari 10. Dengan catatan, kalimat yang dipilih tidak boleh sama dengan langkah praktikum.

In [29]:
subset2=df.iloc[[31, 45, 89, 108, 132],[12]]
subset2

,stemmed_text
31,woahae nya kena efisiensi
45,bmnya bgt efisiensi dongo
89,deret program prioritas menteri pu kena efisie...
108,ih gk had
132,efisiensi


Duplikasikan sheet1 dari template manual. Lalu hitung TF-IDF dari dokumen post test mu.

Terapkan ke dalam kode langkah praktikum, mulai dari poin B. pada poin “mengetahui nilai IDF untuk vocab term tertentu” silahkan pilih 2 vocab bebas berdasarkan dokumenmu.

In [36]:
list_dokumen2 = [item[0] for item in subset2.values.tolist()]
list_dokumen2

['woahae nya kena efisiensi',
 'bmnya bgt efisiensi dongo',
 'deret program prioritas menteri pu kena efisiensi via',
 'ih gk had',
 'efisiensi']

In [37]:
CV = CountVectorizer()
term_fit2 = CV.fit(list_dokumen2)

print(len(term_fit2.vocabulary_))

16


In [38]:
term_frequency_all2 = term_fit2.transform(list_dokumen2)
print(term_frequency_all2)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 20 stored elements and shape (5, 16)>
  Coords	Values
  (0, 4)	1
  (0, 8)	1
  (0, 10)	1
  (0, 15)	1
  (1, 0)	1
  (1, 1)	1
  (1, 3)	1
  (1, 4)	1
  (2, 2)	1
  (2, 4)	1
  (2, 8)	1
  (2, 9)	1
  (2, 11)	1
  (2, 12)	1
  (2, 13)	1
  (2, 14)	1
  (3, 5)	1
  (3, 6)	1
  (3, 7)	1
  (4, 4)	1


In [39]:
# Menghitung nilai TF-IDF untuk dokumen/kalimat tertentu
komentar_term2 = term_fit2.transform(list_dokumen2)
tfidf_transformer2 = TfidfTransformer().fit(komentar_term2)
tfidf2 = tfidf_transformer2.transform(term_frequency_all2)

print(round(tfidf2,3))

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20 stored elements and shape (5, 16)>
  Coords	Values
  (0, 4)	0.327
  (0, 8)	0.468
  (0, 10)	0.58
  (0, 15)	0.58
  (1, 0)	0.549
  (1, 1)	0.549
  (1, 3)	0.549
  (1, 4)	0.309
  (2, 2)	0.379
  (2, 4)	0.213
  (2, 8)	0.306
  (2, 9)	0.379
  (2, 11)	0.379
  (2, 12)	0.379
  (2, 13)	0.379
  (2, 14)	0.379
  (3, 5)	0.577
  (3, 6)	0.577
  (3, 7)	0.577
  (4, 4)	1.0


In [41]:
print(tfidf_transformer2.idf_[term_fit2.vocabulary_['efisiensi']])
print(tfidf_transformer2.idf_[term_fit2.vocabulary_['dongo']])

1.1823215567939547
2.09861228866811


Ada perbedaan yaitu perhitungan manual
